# Session 13 Labs: Search Foundations, Embedding Models & Retrieval Strategy
**AI-Native Software Engineering** · Module 3: Retrieval, Context Engineering & Evaluation
Instructor: Dr. Sandip Chakraborty · TA: Srijib Dey

**How to use this notebook**
1. Click **File > Save a copy in Drive**, so your changes are saved.
2. Run the cells **in order**, from top to bottom (Shift + Enter).
3. For Lab 5, add your OpenAI key in the **Secrets** panel (key icon on the left) with the name `OPENAI_API_KEY`, and switch on *Notebook access*.
4. A CPU runtime is enough for all labs. A GPU (Runtime > Change runtime type) only helps the optional multilingual demo.

Use only the sample data in this notebook. Do not paste real customer data here.

## Setup (run once)

In [ ]:
!pip install -q rank_bm25 sentence-transformers openai
import time, numpy as np
print("Setup done")

### Our sample knowledge base
Eight support FAQs from the Customer Portal running example. All labs reuse this list.

In [ ]:
faqs = ["Reset your password from the login page",
        "Refunds are processed within 7 working days",
        "To complete KYC upload your PAN and Aadhaar",
        "Invalid GSTIN error: check the 15 character format",
        "Refund is delayed if KYC verification is pending",
        "Change your registered mobile number in Settings",
        "Download GST invoice from the Orders page",
        "Cancel an order before it is shipped for full refund"]
for i, f in enumerate(faqs):
    print(i, f)

---
## Lab 1: BM25 from scratch (about 7 min)
**Goal:** rank the FAQs for three queries with BM25 and see which query fails.
*Before running:* predict which of the three queries will fail.

In [ ]:
from rank_bm25 import BM25Okapi

tok = lambda t: t.lower().replace(":", "").split()
bm25 = BM25Okapi([tok(f) for f in faqs])        # defaults: k1=1.5, b=0.75

for q in ["refund delay kyc", "GSTIN error", "how do I get my money back"]:
    scores = bm25.get_scores(tok(q))
    best = scores.argmax()
    print(f"{q!r:30} -> {faqs[best][:40]:40} score={scores[best]:.2f}")

**Discuss:** the money-back query scores 0.00 for every FAQ, yet `argmax` still returns FAQ 0.
That is a silent failure. The cell below adds a minimum score threshold.

In [ ]:
def bm25_answer(q, min_score=0.5):
    scores = bm25.get_scores(tok(q))
    best = scores.argmax()
    return faqs[best] if scores[best] >= min_score else "Sorry, I could not find this."

print(bm25_answer("how do I get my money back"))
print(bm25_answer("GSTIN error"))

---
## Lab 2: Semantic search with Sentence Transformers (about 7 min)
**Goal:** fix the money-back query using embeddings. The model is small (about 90 MB) and runs on CPU.

**Note:** you may see a warning saying *"You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN..."*.
This is harmless and you can ignore it. The model still downloads in a few seconds.

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_emb = model.encode(faqs, normalize_embeddings=True)    # shape (8, 384)
print("Document embeddings:", doc_emb.shape)

queries = ["refund delay kyc", "GSTIN error", "how do I get my money back"]
q_emb = model.encode(queries, normalize_embeddings=True)

scores = util.cos_sim(q_emb, doc_emb)                      # shape (3, 8)
for i, q in enumerate(queries):
    top = scores[i].topk(2)
    print(q)
    for s, j in zip(top.values, top.indices):
        print(f"   {float(s):.2f}  {faqs[int(j)]}")

**Try it:** an English-only model on a Hinglish query. Is the top result sensible?

In [ ]:
qv = model.encode("mera refund kab aayega", normalize_embeddings=True)
best = int(np.argmax(doc_emb @ qv))
print(faqs[best])

### Optional: multilingual model (GPU recommended)
The slides use `intfloat/multilingual-e5-large`. Here we use the smaller `-small` version so it also runs on CPU.
Note the `query:` and `passage:` prefixes, which this model needs.

In [ ]:
mmodel = SentenceTransformer("intfloat/multilingual-e5-small")   # switch to -large on a GPU
docs = ["passage: " + f for f in faqs]
mqueries = ["query: mera refund kab aayega",
            "query: KYC er jonno ki document lagbe",
            "query: मेरा रिफंड कब आएगा?"]
D = mmodel.encode(docs, normalize_embeddings=True)
Q = mmodel.encode(mqueries, normalize_embeddings=True)
for q, row in zip(mqueries, Q @ D.T):
    print(f"{q:40} -> {faqs[int(row.argmax())]}")
print("max_seq_length:", mmodel.max_seq_length)

---
## Lab 3: Hybrid search with Reciprocal Rank Fusion (about 6 min)
**Goal:** combine BM25 and semantic rankings so both the GSTIN query and the money-back query work.

In [ ]:
def rrf(rank_lists, k=60):
    scores = {}
    for ranks in rank_lists:
        for r, doc_id in enumerate(ranks, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + r)
    return sorted(scores, key=scores.get, reverse=True)

def bm25_rank(q, n=5):
    bm = bm25.get_scores(tok(q))
    return [int(i) for i in np.argsort(-bm)[:n] if bm[i] > 0]     # drop zero scores

def vector_rank(q, n=5):
    qv = model.encode(q, normalize_embeddings=True)
    return [int(i) for i in np.argsort(-(doc_emb @ qv))[:n]]

def hybrid_rank(q, n=5):
    return rrf([bm25_rank(q, n), vector_rank(q, n)])

for q in ["GSTIN error", "how do I get my money back", "refund delay kyc"]:
    print(f"{q:28} -> {faqs[hybrid_rank(q)[0]]}")

**Exercise:** add a third ranked list (for example BM25 on only the first 4 words of each FAQ) and pass it to `rrf`.

---
## Lab 4: Reranking with a cross-encoder (about 5 min)
**Goal:** see how a cross-encoder reorders candidates by reading query and document together.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
query = "refund not allowed after shipping?"
candidates = ["Cancel an order before it is shipped for full refund",
              "Refunds are processed within 7 working days",
              "Refund is delayed if KYC verification is pending",
              "Download GST invoice from the Orders page"]

ce_scores = reranker.predict([(query, c) for c in candidates])
print("Cross-encoder ranking:")
for s, c in sorted(zip(ce_scores, candidates), reverse=True):
    print(f"  {s:6.2f}  {c}")

qv = model.encode(query, normalize_embeddings=True)
bi_scores = model.encode(candidates, normalize_embeddings=True) @ qv
print("\nBi-encoder (cosine) ranking, for comparison:")
for s, c in sorted(zip(bi_scores, candidates), reverse=True):
    print(f"  {s:6.2f}  {c}")

---
## Lab 5: How many dimensions do we need? (about 7 min)
**Goal:** shorten OpenAI embeddings (Matryoshka) and watch quality vs. bytes per vector.
Needs `OPENAI_API_KEY` in Colab Secrets. The whole lab costs a tiny fraction of one rupee.

In [ ]:
import os
from openai import OpenAI
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")
client = OpenAI()
print("OpenAI client ready")

In [ ]:
golden5 = [("how do I get my money back", 1), ("which documents for KYC", 2),
           ("GSTIN shows invalid", 3), ("forgot my password", 0),
           ("where is my GST bill", 6), ("stop my order", 7)]

def emb(texts):
    r = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([e.embedding for e in r.data])

D5, Q5 = emb(faqs), emb([q for q, _ in golden5])     # 1536 dims, 2 API calls

def cut(X, d):
    Y = X[:, :d]
    return Y / np.linalg.norm(Y, axis=1, keepdims=True)

for d in [1536, 512, 128, 32, 8, 4]:
    S = cut(Q5, d) @ cut(D5, d).T
    mrr = np.mean([1 / (1 + (S[i] > S[i, c]).sum()) for i, (_, c) in enumerate(golden5)])
    print(f"dims={d:5}  MRR={mrr:.2f}  bytes/vector={d * 4}")

**Discuss:** with only 8 FAQs the task is easy, so quality drops only at very small sizes.
On a real corpus of thousands of similar chunks, the drop starts much earlier.
**Exercise:** add 8 more FAQs of your own (some similar to existing ones) and re-run.

---
## Lab 6: A reusable evaluation harness (about 6 min)
**Goal:** compare BM25, vector, hybrid and hybrid + rerank on a small golden set, with latency.

In [ ]:
golden = [("how do I get my money back", {1, 7}), ("GSTIN error", {3}),
          ("refund delay kyc", {4}), ("forgot my password", {0}),
          ("where is my GST bill", {6}), ("which documents for KYC", {2}),
          ("update phone number", {5}), ("stop my order before delivery", {7})]

def bm25_search(q, top=20):   return bm25_rank(q, top)
def vector_search(q, top=20): return vector_rank(q, top)
def hybrid_search(q, top=20): return hybrid_rank(q, top)[:top]
def rerank_search(q, top=20):
    cands = hybrid_rank(q, 8)
    s = reranker.predict([(q, faqs[i]) for i in cands])
    return [cands[i] for i in np.argsort(-s)][:top]

def evaluate(search_fn, golden, k=1):
    hit, rr, t0 = 0, 0.0, time.perf_counter()
    for q, correct in golden:
        results = search_fn(q, top=20)
        ranks = [i + 1 for i, d in enumerate(results) if d in correct]
        if ranks:
            rr += 1 / ranks[0]
            hit += ranks[0] <= k
    n = len(golden)
    ms = (time.perf_counter() - t0) * 1000 / n
    return {"hit@k": round(hit / n, 2), "mrr": round(rr / n, 2), "ms/query": round(ms, 1)}

systems = {"bm25": bm25_search, "vector": vector_search,
           "hybrid": hybrid_search, "hybrid+rerank": rerank_search}
for name, fn in systems.items():
    print(f"{name:14}", evaluate(fn, golden, k=1))

We use k = 1 here because the corpus has only 8 FAQs; with k = 5 every system would look perfect.
**Exercise:** add 4 Hinglish queries to `golden` and re-run. Which system suffers most?

---
## Optional: pgvector hybrid search on a free hosted PostgreSQL
Colab does not include PostgreSQL. Create a free database on **Neon** (neon.tech) or **Supabase** (supabase.com),
copy its connection string, and save it in Colab Secrets as `DATABASE_URL`.

In [ ]:
!pip install -q "psycopg[binary]" pgvector
import psycopg
from pgvector.psycopg import register_vector
try:
    from google.colab import userdata
    DATABASE_URL = userdata.get("DATABASE_URL")
except Exception:
    from getpass import getpass
    DATABASE_URL = getpass("Paste your DATABASE_URL: ")

conn = psycopg.connect(DATABASE_URL, autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(conn)
conn.execute("DROP TABLE IF EXISTS kb_chunks")
conn.execute("""
CREATE TABLE kb_chunks (
  id serial PRIMARY KEY,
  content text,
  tsv tsvector GENERATED ALWAYS AS (to_tsvector('simple', content)) STORED,
  embedding vector(384))""")
for f, e in zip(faqs, doc_emb):
    conn.execute("INSERT INTO kb_chunks (content, embedding) VALUES (%s, %s)", (f, e))
conn.execute("CREATE INDEX ON kb_chunks USING GIN (tsv)")
conn.execute("CREATE INDEX ON kb_chunks USING hnsw (embedding vector_cosine_ops)")
print("Loaded", conn.execute("SELECT count(*) FROM kb_chunks").fetchone()[0], "rows")

In [ ]:
HYBRID_SQL = """
WITH sem AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY embedding <=> %(qv)s) AS r
  FROM kb_chunks ORDER BY embedding <=> %(qv)s LIMIT 50
), kw AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY ts_rank(tsv, q) DESC) AS r
  FROM kb_chunks, websearch_to_tsquery('simple', %(q)s) q
  WHERE tsv @@ q ORDER BY ts_rank(tsv, q) DESC LIMIT 50
)
SELECT c.content,
       COALESCE(1.0/(60+sem.r), 0) + COALESCE(1.0/(60+kw.r), 0) AS rrf
FROM sem FULL OUTER JOIN kw ON sem.id = kw.id
JOIN kb_chunks c ON c.id = COALESCE(sem.id, kw.id)
ORDER BY rrf DESC LIMIT 3"""

for q in ["GSTIN error", "how do I get my money back"]:
    qv = model.encode(q, normalize_embeddings=True)
    print(q)
    for content, score in conn.execute(HYBRID_SQL, {"qv": qv, "q": q}).fetchall():
        print(f"   {float(score):.4f}  {content}")

---
## Done!
Take-home assignment: build `/v1/search` for the Customer Portal with hybrid search + reranking,
a golden set of 30 queries (including 6 Hinglish/Bengali and 6 with IDs), and compare 2 embedding models and 3 dimensions.